# 👩🏻‍⚕️👨🏻‍⚕️ <span style="color: white; background-color: DarkGreen"><b> Extração do Relatório de Afastamentos </b></span></p>

🧩 <span style="color: MediumSeaGreen"><b> 1- Configuração inicial e controle de processo </b></span></p>
- Importa bibliotecas (Selenium, PyAutoGUI, Pandas, OpenPyXL, logging etc.)  
- Define CONFIG com todos os caminhos (downloads, arquivos movidos, saída, processo)  
- Registra Etapa 0 no arquivo de auditoria PROCESSOS.xlsx, criando rastreabilidade desde o início

🌐 <span style="color: MediumSeaGreen"><b> 2- Acesso automatizado ao Datamace </b></span></p>
- A automação verifica se a janela HTML5 já está aberta → ativa e preenche login  
- Caso contrário → abre navegador via Selenium, acessa o Datamace Cloud e realiza login inicial  
- Em seguida, executa login interno via PyAutoGUI na interface HTML5
- Essa etapa garante acesso completo ao módulo do sistema sem intervenção humana

🏥 <span style="color: MediumSeaGreen"><b> 3- Navegação automática até o módulo de Afastamentos </b></span></p>
- Usando PyAutoGUI, o script:
    - Clica no módulo Medicina Ocupacional  
    - Acessa Ambulatório  
    - Abre a área de Afastamentos  
    - Seleciona os filtros necessários  
    - Configura saída em Excel  
    - Confirma a exportação
- Ou seja, reproduz exatamente a sequência humana de ações na interface

📥 <span style="color: MediumSeaGreen"><b> 4- Confirmação e validação do download </b></span></p>
- O sistema exibe uma caixa de diálogo solicitando “O download da base de Afastamentos foi realizado?”
- Somente após confirmação o pipeline continua, garantindo que haverá arquivo para tratar

🧹 <span style="color: MediumSeaGreen"><b> 5- Tratamento completo do arquivo de Afastamentos </b></span></p>
- O script executa conversão de datas
- Colunas como Data Início, Final, Inclusão etc
- Limpeza do campo Registro
- Truncando e removendo caracteres indesejados
- Conversão do campo Tempo
- Transformando valores textuais (com vírgula ou ponto) em floats numéricos
- Padronização das colunas
- Via um grande dicionário de mapeamento de → para
- Limpeza de valores nulos
- Convertendo NaN → None (compatível com Excel)

📊 <span style="color: MediumSeaGreen"><b> 6- Consolidação de múltiplos arquivos </b></span></p>
- Lista todos arquivos iniciados pelo padrão definido (ex.: “RFA13C”)  
- Processa um por um  
- Adiciona bases válidas à lista principal  
- Move arquivos processados para pasta de arquivos movidos
- Ao final concatena todas as bases  
- Remove duplicidades  
- Registra Etapa 2 no PROCESSOS.xlsx

📁 <span style="color: MediumSeaGreen"><b> 7- Geração da planilha final formatada </b></span></p>
- Cria AFASTAMENTOS.xlsx
- Com aba única  
- Tabela Excel estruturada  
- Estilo TableStyleLight13  
- Dados limpos e normalizados

🔄 <span style="color: MediumSeaGreen"><b> 8- Atualização automática da planilha Controle HC e Atestados </b></span></p>
- Abre o arquivo
- Navega para a aba/célula correta usando Ctrl + G  
- Executa o comando de refresh (Alt + F5)
- Atualizando automaticamente os controles corporativos dependentes dessa base

🗃️ <span style="color: MediumSeaGreen"><b> 9- Exportação final para banco de dados </b></span></p>
- Gera automaticamente tb_afastamentos.csv
- Para utilização em Pipelines SQL, Data Warehouse, Dashboards e Modelo de People Analytics

⏱️ <span style="color: MediumSeaGreen"><b> 10- Resumo final </b></span></p>
- Ao final o script apresenta:
    - Status → Processo finalizado  
    - Tempo de execução total  
    - Informações de auditoria registradas

# Importando as Bibliotecas

In [1]:
import logging
import os
import pandas as pd
import pyautogui
import pygetwindow as gw
import pyperclip
import shutil
import socket
import sys
import time
import tkinter as tk
import win32com.client
import win32gui, win32con
from asyncio.log import logger
from contextlib import contextmanager
from datetime import datetime, date
from dotenv import load_dotenv
from openpyxl import load_workbook, Workbook
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.worksheet.table import Table, TableStyleInfo
from pathlib import Path
from selenium import webdriver
from selenium.common.exceptions import WebDriverException
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver import ActionChains
from tkinter import simpledialog
from webdriver_manager.chrome import ChromeDriverManager

tempo_0 = [id, datetime.today(), 0]

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Processo de Importação finalizado')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

   ✅ Processo de Importação finalizado

----------------------------------------------------------------------------------------------------


# Caminhos de Pastas

In [2]:
CONFIG = {
    'id_processo': 2,
    'processos': Path(r'X:\Gestão de Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\PROCESSOS.xlsx'),
    'origem': Path(r'C:\Users\rodrigo.bernandes\Downloads'),
    'movidos': Path(r'X:\Gestão de Pessoas\Analytics\03 - Bases\2. ARQUIVOS MOVIDOS'),
    'saida': Path(r'X:\Gestão de Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\AFASTAMENTOS.xlsx'),
    'env_path': Path(r'X:\Gestão de Pessoas\Analytics\08 - Notebooks Python\08.3 - Automações\.env'),
    'padrao': 'AFASTA',
}

# Registra Etapa do Processo

In [3]:
def append_registro_processo(caminho, id_proc, etapa):
    try:
        wb = load_workbook(caminho)
        ws = wb['REGISTROS']
        ws.append([id_proc, datetime.today(), etapa])
        wb.save(caminho)
        wb.close()
    except Exception as e:
        print(f"❌ Erro ao registrar etapa {etapa}: {e}")

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Registro de processos')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

   ✅ Registro de processos

----------------------------------------------------------------------------------------------------


# Configurações Iniciais e Acessando o Datamace

In [4]:
load_dotenv(dotenv_path='env_path')

cloud_user = os.getenv("CLOUD_USER")
cloud_password = os.getenv("CLOUD_PASSWORD")
app_user = os.getenv("APP_USER")
app_password = os.getenv("APP_PASSWORD")
datamace = os.getenv("SITE_DATAMACE")

# Validar se todas existem
if not all([cloud_user, cloud_password, app_user, app_password]):
    raise ValueError("Uma ou mais variáveis de ambiente não foram encontradas. Verifique o arquivo .env.")

titulo_aba = "HTML5"

# Verifica se janela 'HTML5' está aberta
if gw.getWindowsWithTitle(titulo_aba):
    pyautogui.click(836, 445)
    time.sleep(0.5)
    pyautogui.typewrite(app_user)
    pyautogui.press('tab')
    time.sleep(0.5)
    pyautogui.typewrite(app_password)
    pyautogui.press('enter')
else:
    driver = webdriver.Chrome()
    time.sleep(1)
    window = win32gui.GetForegroundWindow()
    win32gui.ShowWindow(window, win32con.SW_MAXIMIZE)
    time.sleep(1)
    driver.get(datamace)
    time.sleep(8)
    # Login cloud
    driver.find_element(By.NAME, "username").send_keys(cloud_user)
    driver.find_element(By.NAME, "Password").send_keys(cloud_password)
    driver.find_element(By.NAME, "Password").send_keys(Keys.ENTER)
    time.sleep(30)
    # Login pyautogui
    pyautogui.click(836, 445)
    time.sleep(0.5)
    pyautogui.typewrite(app_user)
    pyautogui.press('tab')
    time.sleep(0.5)
    pyautogui.typewrite(app_password)
    pyautogui.press('enter')

print("🏁 Acesso finalizado")

🏁 Acesso finalizado


# Acessando o MO

In [5]:
# Fechando janelas de Novidades do Datamace
time.sleep(3)
pyautogui.press('enter')
time.sleep(3)
pyautogui.click(x=1268, y=199)
# Acessando o MO
time.sleep(10) # Tempo maior para aguardar carregar o MO
pyautogui.click(x=474, y=196) # Clicando no módulo MO
time.sleep(3)
pyautogui.click(x=1079, y=235) # Fecha a janela de Tarefas Anuais
time.sleep(2)
pyautogui.click(x=52, y=299) # Ambulatório
time.sleep(1)
pyautogui.click(x=163, y=482) # Afastamentos
time.sleep(1)
pyautogui.click(x=356, y=477) # Rel. de Afastamentos
time.sleep(2)
pyautogui.click(x=604, y=274) # Multi processamento
time.sleep(2)
pyautogui.click(x=606, y=313) # Todos
time.sleep(2)
pyautogui.click(x=725, y=570) # Período da pesquisa
time.sleep(2)
pyautogui.typewrite("01011960", interval=0.05) # Data para extrair todos os registros
time.sleep(2)
pyautogui.click(x=755, y=592) # Situação
time.sleep(2)
pyautogui.click(x=737, y=640) # Todos
time.sleep(2)
pyautogui.click(x=749, y=671) # Gera arquivo em Excel?
time.sleep(2)
pyautogui.click(x=735, y=701) # Sim
time.sleep(1)
pyautogui.press("tab")
pyautogui.press("enter")
time.sleep(2)
pyautogui.click(x=564, y=708) # Confirmar
time.sleep(2)

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Extraindo relatório')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

   ✅ Extraindo relatório

----------------------------------------------------------------------------------------------------


# Aguardando a Conclusão da Base AFASTADOS

In [6]:
# Configuração de logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

def verificar_download_base() -> bool:
    """
    Exibe uma caixa de diálogo para confirmar se o download da base COLAB foi realizado.
    """
    tipo_escolhido = pyautogui.confirm(
        text='Foi realizado o download da base AFASTADOS?', 
        title='Seleção de Extração', 
        buttons=['Sim']
    )

    # Se fechar a janela
    if tipo_escolhido is None:
        logging.warning("Operação cancelada pelo usuário no prompt inicial. ❌ Encerrando o script.")
        sys.exit(0)
        
    # Se for 'Sim'
    logging.info(f"Opção selecionada: {tipo_escolhido}. ✅ Continuando o processo...")
    return True

# --- Execução principal ---
if __name__ == "__main__":
    # Chama a função de validação antes de seguir com o restante do código
    verificar_download_base()
    
    logging.info("Iniciando a leitura e processamento dos dados...")

2026-06-25 09:25:06,850 - INFO - Opção selecionada: Sim. ✅ Continuando o processo...
2026-06-25 09:25:06,872 - INFO - Iniciando a leitura e processamento dos dados...


# Mapeamento de Colunas

In [7]:
MAPEAMENTO_COLUNAS = {
    'EMPRESA': 'cod_empresa',
    'REGISTRO': 'registro',
    'NOME': 'nome',
    'DATA ADM': 'data_admissao',
    'C.P.F': 'cpf',
    'N.I.T': 'nit',
    'CODIGO': 'cod_afastamento',
    'DESCRICAO DO AFASTAMENTO': 'descricao_afastamento',
    'SEFIP': 'sefip',
    'DATA AFA': 'data_afastamento',
    'DATA RET': 'data_retorno',
    'DIAS AFA': 'dias_afastados',
    'SALARIO': 'salario',
    'MDO': 'mdo',
    'OBSERVACOES': 'observacoes',
    'ATESTADO': 'atestado',
    'BENEFICIO PREVIDENCIARIO': 'beneficio_previdenciario',
    'DIA TRABALHADO': 'dia_trabalhado',
    'C.I.D.': 'cid',
}
COLUNAS_DATA = ['DATA ADM', 'DATA AFA', 'DATA RET']

# Processando Afastamentos

In [8]:
@contextmanager

def gerenciar_workbook(caminho: Path, sheet: str):
    """Context manager para operações seguras com workbook."""
    wb = load_workbook(caminho)
    ws = wb[sheet]
    try:
        yield ws
    finally:
        wb.save(caminho)

def registrar_etapa(caminho: Path, id_proc: int, etapa: int):
    """Registra etapa do processo."""
    with gerenciar_workbook(caminho, 'REGISTROS') as ws:
        ws.append([id_proc, datetime.today(), etapa])
    logger.debug(f"✅ Etapa {etapa} registrada")

def converter_datas(df: pd.DataFrame, colunas: list) -> pd.DataFrame:
    """Converte colunas para datetime."""
    for col in colunas:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], format='%d/%m/%Y', errors='coerce')
    return df

def limpar_registro(df: pd.DataFrame) -> pd.DataFrame:
    """Limpa coluna REGISTRO (pega apenas primeiros 6 caracteres)."""
    if 'REGISTRO' in df.columns:
        df['REGISTRO'] = df['REGISTRO'].astype(str).str[:6]
        df['REGISTRO'] = pd.to_numeric(df['REGISTRO'], errors='coerce')
    return df

def limpar_observacoes(df: pd.DataFrame) -> pd.DataFrame:
    """Remove espaços em branco da coluna 'observacoes'."""
    # Tenta ambas as variações de case
    col_name = 'observacoes' if 'observacoes' in df.columns else 'OBSERVACOES'
    if col_name in df.columns:
        df[col_name] = df[col_name].fillna('').str.strip()
        df[col_name] = df[col_name].str.replace(r'\s+', ' ', regex=True)
    return df

def mapear_colunas_afastamentos(df: pd.DataFrame) -> pd.DataFrame:
    """Mapeia colunas do arquivo de afastamentos."""
    colunas_existentes = {k: v for k, v in MAPEAMENTO_COLUNAS.items() if k in df.columns}
    return df.rename(columns=colunas_existentes)[list(colunas_existentes.values())]

def processar_arquivo(caminho: Path) -> pd.DataFrame:
    """Processa um arquivo de afastamentos."""
    try:
        logger.debug(f"Carregando: {caminho.name}")
        
        # Carregar
        df = pd.read_excel(caminho, engine='openpyxl')
        
        # Limpar
        df = df.dropna(subset=['NOME'])
        
        # Converter tipos
        df = limpar_registro(df)
        df = converter_datas(df, COLUNAS_DATA)    
        df = limpar_observacoes(df)
        
        # Mapear colunas
        df = mapear_colunas_afastamentos(df)
        
        logger.debug(f"✅ {len(df)} registros processados")
        return df
        
    except Exception as e:
        logger.error(f"❌ Erro ao processar {caminho.name}: {e}")
        return None
    
def criar_excel_com_tabela(df: pd.DataFrame, caminho: Path):
    """Cria Excel com tabela formatada."""
    logger.info(f"📝 Criando Excel final ({len(df)} registros)...")
    
    # Salvar com Pandas
    df.to_excel(caminho, sheet_name='AFASTAMENTOS', index=False, engine='openpyxl')
    
    # Aplicar formatação
    wb = load_workbook(caminho)
    ws = wb.active
    
    tabela = Table(displayName="AFASTAMENTOS", ref=ws.dimensions)
    estilo = TableStyleInfo(
        name="TableStyleLight13",
        showFirstColumn=False,
        showLastColumn=False,
        showRowStripes=True,
        showColumnStripes=True
    )
    tabela.tableStyleInfo = estilo
    ws.add_table(tabela)
    
    wb.save(caminho)
    logger.info(f"✅ Excel criado: {caminho}")
# Main
if __name__ == "__main__":
    tempo_inicio = datetime.now()
    
    logger.info("=" * 80)
    logger.info("🔄 PROCESSAMENTO DE AFASTAMENTOS")
    logger.info("=" * 80)

2026-06-25 09:25:06,968 - INFO - ================================================================================
2026-06-25 09:25:06,969 - INFO - 🔄 PROCESSAMENTO DE AFASTAMENTOS
2026-06-25 09:25:06,970 - INFO - ================================================================================


# Processamento e Consolidação do Arquivo

In [9]:
try:
    # Etapa 1: Registrar início
    registrar_etapa(CONFIG['processos'], CONFIG['id_processo'], 0)
        
    # Etapa 2: Buscar arquivos
    logger.info("\n📂 Buscando arquivos...")
    arquivos = sorted([f for f in CONFIG['origem'].iterdir() if f.name.startswith(CONFIG['padrao'])])
        
    if not arquivos:
        logger.warning("❌ Nenhum arquivo encontrado")
        exit()
        
    logger.info(f"✅ {len(arquivos)} arquivo(s) encontrado(s)\n")
        
    registrar_etapa(CONFIG['processos'], CONFIG['id_processo'], 1)
        
    # Etapa 3: Processar arquivos (CONSOLIDAR FORA DO LOOP)
    logger.info("🔄 Processando arquivos...\n")
        
    todas_bases = []
        
    for idx, arquivo in enumerate(arquivos, 1):
        logger.info(f"[{idx}/{len(arquivos)}] {arquivo.name}")
            
        df = processar_arquivo(arquivo)
            
        if df is not None and len(df) > 0:
            todas_bases.append(df)
            logger.info(f"   ✅ {len(df)} registros adicionados")
                
            try:
                destino = CONFIG['movidos'] / arquivo.name
                shutil.move(str(arquivo), str(destino))
                logger.info(f"   ✅ Movido para arquivos processados\n")
            except Exception as e:
                logger.error(f"   ⚠️ Erro ao mover: {e}\n")
        else:
            logger.warning(f"   ❌ Sem dados válidos\n")
        
    registrar_etapa(CONFIG['processos'], CONFIG['id_processo'], 2)
        
    # Etapa 4: Consolidar e salvar (UMA ÚNICA VEZ)
    if todas_bases:
        logger.info("💾 Consolidando dados...")
        base_final = pd.concat(todas_bases, ignore_index=True)
        base_final = base_final.drop_duplicates()
        logger.info(f"✅ {len(base_final)} registros consolidados\n")
            
        criar_excel_com_tabela(base_final, CONFIG['saida'])
    else:
        logger.error("❌ Nenhum arquivo foi processado!")
        exit()
        
    # Finalizar
    tempo_duracao = datetime.now() - tempo_inicio
        
    logger.info("\n" + "=" * 80)
    logger.info("✅ PROCESSO FINALIZADO COM SUCESSO!")
    logger.info(f"   Tempo de execução: {tempo_duracao}")
    logger.info("=" * 80)
        
except Exception as e:
    logger.error(f"\n❌ ERRO CRÍTICO: {e}")
    import traceback
    logger.error(traceback.format_exc())

2026-06-25 09:25:07,441 - INFO - 
📂 Buscando arquivos...
2026-06-25 09:25:07,442 - INFO - ✅ 1 arquivo(s) encontrado(s)

2026-06-25 09:25:07,742 - INFO - 🔄 Processando arquivos...

2026-06-25 09:25:07,742 - INFO - [1/1] AFASTA-010160-31129909231944.XLSX
c:\Users\rodrigo.bernandes\AppData\Local\Programs\Python\Python312\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
2026-06-25 09:25:17,613 - INFO -    ✅ 37209 registros adicionados
2026-06-25 09:25:17,770 - INFO -    ✅ Movido para arquivos processados

2026-06-25 09:25:18,109 - INFO - 💾 Consolidando dados...
2026-06-25 09:25:18,155 - INFO - ✅ 37209 registros consolidados

2026-06-25 09:25:18,156 - INFO - 📝 Criando Excel final (37209 registros)...
2026-06-25 09:25:46,594 - INFO - ✅ Excel criado: X:\Gestão de Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\AFASTAMENTOS.xlsx
2026-06-25 09:25:46

# Atualizando o Arquivo Excel Controle HC e Atestados

In [10]:
# Caminho do arquivo
path_excel = r"X:\Gestão de Pessoas\Analytics\10 - Relatórios\10.4 - HC e Atestados Médicos\Controle_HC e Atestados.xlsx"
os.startfile(path_excel) # Abre o arquivo
time.sleep(60)
pyautogui.press('esc')
time.sleep(15)

# Utiliza o comando "Ir para" (Ctrl + G) para navegar até a aba e célula
pyautogui.hotkey('ctrl', 'g')
time.sleep(1)

# Digita o endereço completo
pyautogui.write('AFASTADOS!B5')
time.sleep(1)
pyautogui.press('enter')
time.sleep(3)

#Atualizando o arquivo
pyautogui.hotkey('alt', 'F5')
time.sleep(2)

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Planilha atualizada com sucesso')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

   ✅ Planilha atualizada com sucesso

----------------------------------------------------------------------------------------------------


# Criando Base em CSV para o Banco de Dados

In [11]:
# Ler o XLSX
df = pd.read_excel(r'X:\Gestão de Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\AFASTAMENTOS.xlsx')

# Salvar como CSV
df.to_csv(r'X:\Gestão de Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\tb_afastamentos.csv', index=False, encoding='utf-8')

print("✅ Arquivo convertido para CSV com sucesso!")

✅ Arquivo convertido para CSV com sucesso!


# Resumo de Finalização do Processo

In [12]:
tempo_1 = [id, datetime.today(), 1]

print('----------------------------------------------------------------------------------------------------')
print('')
print('     ✅  Processo finalizado')
print('')
print('     ⏱️   Tempo de execução:')
print('')
print(f'   {tempo_1[1] - tempo_0[1]}')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

     ✅  Processo finalizado

     ⏱️   Tempo de execução:

   0:22:11.706553

----------------------------------------------------------------------------------------------------
